In [ ]:
import pandas as pd

test = pd.read_csv("../dataset/house-prices-advanced-regression-techniques/test.csv")
test.head()

In [ ]:
train = pd.read_csv("../dataset/house-prices-advanced-regression-techniques/train.csv")
train.head()

In [ ]:
# Menampilkan ringkasan informasi dari dataset
train.info()

In [ ]:
# Menampilkan statistik deskriptif dari dataset
train.describe(include="all")

In [ ]:
# Memeriksa jumlah nilai yang hilang di setiap kolom
missing_values = train.isnull().sum()
missing_values[missing_values > 0]

In [ ]:
# Memisahkan kolom yang memiliki missing value lebih dari 75% dan kurang dari 75%
less = missing_values[missing_values < 1000].index
over = missing_values[missing_values > 1000].index

In [ ]:
# Mengisi nilai yang hilang dengan median untuk kolam numerik
numeric_features = train[less].select_dtypes(include=['number']).columns
train[numeric_features] = train[numeric_features].fillna(train[numeric_features].median())

In [ ]:
# Mengisi nilai yang hilang dengan modus ntuk kolom kategori
kategorical_features = train[less].select_dtypes(include=['object']).columns

for column in kategorical_features:
  train[column] = train[column].fillna(train[column].mode()[0])

In [ ]:
# Menghapus kolom dengan terlalu banyak nilai yang hilang
df = train.drop(columns=over)

In [ ]:
# Verifikasi missing value (pemeriksaan data setelah cleaning)
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

In [ ]:
# Check outlier dengan metode IQR (Interquartile Range) -> rentang antara Q1 dan Q3 dalam data. Nilai yang terletak di luar batas IQR dianggap sebagai outlier

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

for feature in numeric_features:
  plt.figure(figsize=(10, 6))
  sns.boxplot(x=df[feature])
  plt.title(f'Box Plot of {feature}')
  plt.show()

In [ ]:
# Contoh sederhana untuk mengidentifikasi outliers menggunakan IQR
Q1 = df[numeric_features].quantile(0.25)
Q3 = df[numeric_features].quantile(0.75)
IQR = Q3 - Q1

# Filter dataframe untuk hanya menyimpan baris yang tidak mengandung outliers pada kolom numerik
condition = ~((df[numeric_features] < (Q1 - 1.5 * IQR)) | (df[numeric_features] > (Q3 + 1.5 * IQR))).any(axis=1)
df_filtered_numeric = df.loc[condition, numeric_features]

# Menggabungkan kembali dengan kolom kategorikal
categorical_features = df.select_dtypes(include=['object']).columns
df = pd.concat([df_filtered_numeric, df.loc[condition, categorical_features]], axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler
 
# Standardisasi fitur numerik
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

In [ ]:
# Histogram sebelum standardisasi
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(train[numeric_features[3]], kde=True)
plt.title("Histogram sebelum standardisasi")